In [31]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sentence_transformers import SentenceTransformer
import warnings

warnings.filterwarnings("ignore")

In [41]:
def load_data():
    conn = sqlite3.connect('../../data/logs.db')
    df = pd.read_sql_query("SELECT * FROM logs", conn)
    conn.close()
    return df

In [11]:
load_data() 

,id,timestamp,log_type,details,status
0,1,2026-06-02 12:20:09,AUTH,"{'ts': '2026-06-02 12:20:09', 'user': 'guest',...",SUCCESS
1,2,2024-01-01T00:00:00,WEB,"{'ts': '2024-01-01T00:00:00', 'ip': '1.2.3.4',...",200
2,3,2026-06-02 12:20:11,DATABASE,"{'ts': '2026-06-02 12:20:11', 'level': 'INFO',...",N/A
3,4,2026-06-02 12:20:12,DATABASE,"{'ts': '2026-06-02 12:20:12', 'level': 'INFO',...",N/A
4,5,2026-06-02 12:22:57,DATABASE,"{'ts': '2026-06-02 12:22:57', 'level': 'ERROR'...",N/A
...,...,...,...,...,...
927,928,2026-06-02 20:45:44,DATABASE,"{'ts': '2026-06-02 20:45:44', 'level': 'ERROR'...",N/A
928,929,2026-06-02 20:45:45,WEB,"{'ts': '2026-06-02 20:45:45', 'ip': '10.0.0.5'...",200
929,930,2026-06-02 20:45:45,DATABASE,"{'ts': '2026-06-02 20:45:45', 'level': 'ERROR'...",N/A
930,931,2026-06-02 20:45:47,WEB,"{'ts': '2026-06-02 20:45:47', 'ip': '237.84.2....",200


In [42]:
def run_semantic_detector(df):
    print("⏳ Running Detector 1: Semantic Pipeline (MiniLM -> PCA -> IF)...")

    # Step 1: Generate embeddings using MiniLM
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(df['details'].tolist(), show_progress_bar=True)

    print("✅ Embeddings generated.\n")
    print(embeddings)

    # Step 2: Dimensionality reduction with PCA
    n_comp = min(20,len(df))
    pca = PCA(n_components=n_comp, random_state=42)
    compressed_embeddings = pca.fit_transform(embeddings)

    print(f"✅ PCA completed. Reduced to {n_comp} dimensions.\n")
    print(compressed_embeddings)

    # Step 3: Anomaly detection with Isolation Forest
    iso_forest = IsolationForest(contamination=0.02, random_state=42)
    return iso_forest.fit_predict(compressed_embeddings)

    

In [ ]:
#print 10 rows of the dataframe from logs.db
connection = sqlite3.connect('../../data/logs.db')
df = pd.read_sql_query("SELECT * FROM logs", connection)
connection.close()
print(df.head(10))

   id            timestamp  log_type  \
0   1  2026-06-02 12:20:09      AUTH   
1   2  2024-01-01T00:00:00       WEB   
2   3  2026-06-02 12:20:11  DATABASE   
3   4  2026-06-02 12:20:12  DATABASE   
4   5  2026-06-02 12:22:57  DATABASE   
5   6  2026-06-02 12:22:57      AUTH   
6   7  2026-06-02 12:22:59  DATABASE   
7   8  2026-06-02 12:23:01      AUTH   
8   9  2026-06-02 12:23:02       WEB   
9  10  2026-06-02 12:23:03       WEB   

                                             details   status  
0  {'ts': '2026-06-02 12:20:09', 'user': 'guest',...  SUCCESS  
1  {'ts': '2024-01-01T00:00:00', 'ip': '1.2.3.4',...      200  
2  {'ts': '2026-06-02 12:20:11', 'level': 'INFO',...      N/A  
3  {'ts': '2026-06-02 12:20:12', 'level': 'INFO',...      N/A  
4  {'ts': '2026-06-02 12:22:57', 'level': 'ERROR'...      N/A  
5  {'ts': '2026-06-02 12:22:57', 'user': 'guest',...  SUCCESS  
6  {'ts': '2026-06-02 12:22:59', 'level': 'INFO',...      N/A  
7  {'ts': '2026-06-02 12:23:01', 'user': 'admin

In [15]:
details = (df['details'].tolist())
print(details[:10])

["{'ts': '2026-06-02 12:20:09', 'user': 'guest', 'status': 'SUCCESS'}", "{'ts': '2024-01-01T00:00:00', 'ip': '1.2.3.4', 'path': '/', 'status': '200'}", "{'ts': '2026-06-02 12:20:11', 'level': 'INFO', 'msg': 'Connection successful'}", "{'ts': '2026-06-02 12:20:12', 'level': 'INFO', 'msg': 'Connection successful'}", "{'ts': '2026-06-02 12:22:57', 'level': 'ERROR', 'msg': 'Connection timeout exceeded'}", "{'ts': '2026-06-02 12:22:57', 'user': 'guest', 'status': 'SUCCESS'}", "{'ts': '2026-06-02 12:22:59', 'level': 'INFO', 'msg': 'Connection successful'}", "{'ts': '2026-06-02 12:23:01', 'user': 'admin', 'status': 'SUCCESS'}", "{'ts': '2026-06-02 12:23:02', 'ip': '10.0.0.5', 'path': '/api/v1/login', 'status': '200'}", "{'ts': '2026-06-02 12:23:03', 'ip': '237.84.2.178', 'path': '/api/v1/search', 'status': '200'}"]


In [16]:
run_semantic_detector(load_data())

⏳ Running Detector 1: Semantic Pipeline (MiniLM -> PCA -> IF)...


Batches: 100%|██████████| 30/30 [00:05<00:00,  5.67it/s]


✅ Embeddings generated.

[[-0.00768412  0.1022056  -0.00264616 ...  0.09317171 -0.05223896
   0.00770177]
 [-0.01709467  0.06765962 -0.03176519 ...  0.07480035 -0.04260188
  -0.01415239]
 [-0.04732236  0.04007109 -0.00624042 ...  0.11276086 -0.0430159
  -0.05239982]
 ...
 [-0.04171121  0.03344827 -0.01569448 ...  0.09050117 -0.05178159
  -0.07245066]
 [-0.0180137   0.07336812 -0.01985126 ...  0.08049509 -0.02517484
  -0.01095731]
 [-0.07008849  0.05542609 -0.04536855 ...  0.0931288  -0.03785498
   0.0095757 ]]
✅ PCA completed. Reduced to 20 dimensions.

[[ 0.22432236  0.26482671 -0.00134936 ...  0.00664371 -0.0030166
  -0.00169663]
 [-0.17254892  0.03995989 -0.12759076 ... -0.01572558 -0.08714157
   0.07494614]
 [ 0.16319177 -0.25037882 -0.07237732 ... -0.00655794  0.00886028
  -0.00548128]
 ...
 [ 0.05152138 -0.36794114  0.3990879  ...  0.00930446 -0.01289643
   0.01352602]
 [-0.21731155  0.04146636 -0.1823522  ...  0.00779448 -0.02907548
  -0.00367113]
 [-0.24223152 -0.03365425 -0.07

0      1
1     -1
2      1
3      1
4      1
      ..
927    1
928   -1
929    1
930    1
931    1
Name: anomaly_score, Length: 932, dtype: int64

In [50]:
def run_behavioral_detector(df):
    print("⏳ Running Detector 2: Behavioral Pipeline (Metadata + Velocity)...")

    # 1. Feature extraction
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df['invalid_timestamp'] = df['timestamp'].isna().astype(int)
    df['hour'] = df['timestamp'].dt.hour.fillna(-1).astype(int)

    # 2. MAP BEHAVIORAL FREQUENCY (VELOCITY)
    valid_mask = df['timestamp'].notna()
    valid_df = df[valid_mask].copy()
  
    # Sort chronologically and explicitly set 'id' as the index
    valid_df = valid_df.sort_values(['timestamp', 'id']).set_index('id')
    
    # Calculate velocity
    velocity = valid_df.groupby('log_type').rolling('10s', on='timestamp')['hour'].count()
    
    # Drop the 'log_type' group label, leaving a Series where the index is the unique 'id'
    velocity_series = velocity.reset_index(level=0, drop=True)
    
  
 
    # Strips out any duplicate indices caused by dirty SQLite data or Pandas bugs,
    # ensuring the .map() function never crashes.
    
    velocity_series = velocity_series[~velocity_series.index.duplicated(keep='last')]
    
    # Step E: Map the velocities back to the main DataFrame safely!
    df['velocity_10s'] = df['id'].map(velocity_series).fillna(0)

    # 3. Encode categorical features (One-Hot Encoding)
    ohe_features = pd.get_dummies(df[['log_type', 'status']], drop_first=False).astype(int)
    num_features = df[['hour', 'invalid_timestamp', 'velocity_10s']]
    
    # 4. Build feature matrix
    features = pd.concat([num_features, ohe_features], axis=1)
    features = features.fillna(0)

    # 5. Isolation Forest
    iso_forest = IsolationForest(contamination=0.02, random_state=42)
    return iso_forest.fit_predict(features)

In [18]:
run_behavioral_detector(load_data())

⏳ Running Detector 2: Behavioral Pipeline ( Metadata -> Isolation Forest)...
✅ Isolation Forest applied. Anomaly scores assigned.



0     -1
1     -1
2      1
3      1
4      1
      ..
927    1
928    1
929    1
930    1
931    1
Name: anomaly_score, Length: 932, dtype: int64

In [44]:
def aggregate_scores(row):
    # -1 is Anomaly, 1 is Normal in scikit-learn
    sem_anom = row['semantic_pred'] == -1
    beh_anom = row['behavioral_pred'] == -1
    
    if sem_anom and beh_anom:
        return "High Risk"
    elif sem_anom or beh_anom:
        return "Medium Risk"
    else:
        return "Normal"

In [51]:
def main():
    df = load_data()
    if len(df) < 50:  # Arbitrary threshold for minimum data
        print("⚠️ Waiting for more logs to train PCA effectively...")
        return
        
    # Run Detectors
    df['semantic_pred'] = run_semantic_detector(df)
    df['behavioral_pred'] = run_behavioral_detector(df)

    # Debug: Print predictions
    print("\n🔍 Semantic Detector Predictions:")
    print(df['semantic_pred'].value_counts())
    print("\n🔍 Behavioral Detector Predictions:")
    print(df['behavioral_pred'].value_counts())
    
    
    
    # Aggregate
    df['risk_level'] = df.apply(aggregate_scores, axis=1)
    
    print("\n🔍 Risk Lavel:")
    print(df['risk_level'])
    
    # Filter and Display Results
    risks = df[df['risk_level'] != 'Normal']
    
    print("\n==================================================")
    print("📊 ENSEMBLE ANOMALY DETECTION REPORT")
    print("==================================================")
    
    if risks.empty:
        print("✅ System normal. No anomalies detected.")
    else:
        for _, row in risks.iterrows():
            print(f"\n🚨 [{row['risk_level'].upper()}]")
            print(f"ID: {row['id']} | Type: {row['log_type']} | TS: {row['timestamp']}")
            print(f"Message: {row['details']}")
            
            # Print the specific trigger
            if row['semantic_pred'] == -1:
                print("   -> Trigger: Unusual Semantic Pattern")
            if row['behavioral_pred'] == -1:
                print("   -> Trigger: Unusual Behavioral Metadata")

if __name__ == "__main__":
    main()

⏳ Running Detector 1: Semantic Pipeline (MiniLM -> PCA -> IF)...


Batches: 100%|██████████| 30/30 [00:04<00:00,  6.34it/s]


✅ Embeddings generated.

[[-0.00768412  0.1022056  -0.00264616 ...  0.09317171 -0.05223896
   0.00770177]
 [-0.01709467  0.06765962 -0.03176519 ...  0.07480035 -0.04260188
  -0.01415239]
 [-0.04732236  0.04007109 -0.00624042 ...  0.11276086 -0.0430159
  -0.05239982]
 ...
 [-0.04171121  0.03344827 -0.01569448 ...  0.09050117 -0.05178159
  -0.07245066]
 [-0.0180137   0.07336812 -0.01985126 ...  0.08049509 -0.02517484
  -0.01095731]
 [-0.07008849  0.05542609 -0.04536855 ...  0.0931288  -0.03785498
   0.0095757 ]]
✅ PCA completed. Reduced to 20 dimensions.

[[ 0.22432236  0.26482671 -0.00134936 ...  0.00664371 -0.0030166
  -0.00169663]
 [-0.17254892  0.03995989 -0.12759076 ... -0.01572558 -0.08714157
   0.07494614]
 [ 0.16319177 -0.25037882 -0.07237732 ... -0.00655794  0.00886028
  -0.00548128]
 ...
 [ 0.05152138 -0.36794114  0.3990879  ...  0.00930446 -0.01289643
   0.01352602]
 [-0.21731155  0.04146636 -0.1823522  ...  0.00779448 -0.02907548
  -0.00367113]
 [-0.24223152 -0.03365425 -0.07